In [1]:
import pandas as pd
import numpy as np


In [2]:
data = pd.read_csv('ks.csv')
data_original = data.copy()

In [3]:
data.head(10)

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Состояние,Инвесторов,Страна,Собрано в долларах,Цель в долларах
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,successful,23,US,600.00,600.00
1,Arcade County (Canceled),Games,Games,USD,2012-04-29,2012-03-30 23:40:45,canceled,5,US,71.00,9000.00
2,Hayashi Skate Co. Solar Skateboard backpack,Accessories,Fashion,CAD,2017-07-22,2017-05-23 23:00:13,canceled,8,CA,360.36,2391.77
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,failed,20,US,502.00,10000.00
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,successful,62,US,2414.00,1400.00
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03 16:17:21,successful,86,US,10030.88,10000.00
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12 05:19:53,failed,0,US,0.00,10000.00
7,The Wild World of Ogots - YOU Become an Ogot (...,Publishing,Publishing,USD,2017-12-20,2017-11-20 23:06:11,successful,45,US,1305.00,1075.00
8,"VIGATAR, the fitness-machine that lets you pla...",Hardware,Technology,EUR,2014-08-28,2014-06-29 23:30:46,failed,22,NL,4395.53,131879.15
9,Escape Room Palm Springs - Real Life Escape Game,Live Games,Games,USD,2015-09-21,2015-08-21 17:34:28,failed,88,US,2553.00,7500.00


In [4]:
data.shape

(378661, 11)

In [5]:
### Оставим только нужные состояния
data = data_original
data = data[data['Состояние'].isin(['failed', 'successful'])]
# data['Состояние'] = data['Состояние'].map({'failed': 0, 'successful': 1})
# data['Состояние'].value_counts()


In [6]:
data.head(3)

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Состояние,Инвесторов,Страна,Собрано в долларах,Цель в долларах
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,successful,23,US,600.0,600.0
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,failed,20,US,502.0,10000.0
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,successful,62,US,2414.0,1400.0


### Что берем за таргет? 2 варианта:

- Будем решать задачу классификации, разметив объекты следующим образом: те проекты, у которых успешный статус, единичкой, а остальные ноликами

- Можем предсказывать просто собранное количество денег, применять модель, а потом уже смотреть, нужная ли сумма получилось. Тогда мы решаем задачу регрессии.

In [7]:
### Создадим колонку таргет1 (бинарный таргет)
data.loc[data['Состояние'] == 'failed', 'таргет1'] = 0
data.loc[data['Состояние'] == 'successful', 'таргет1'] = 1

In [8]:
data = data.drop('Состояние', axis=1)
data.head(10)

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,Собрано в долларах,Цель в долларах,таргет1
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,23,US,600.00,600.00,1.0
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,20,US,502.00,10000.00,0.0
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,62,US,2414.00,1400.00,1.0
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03 16:17:21,86,US,10030.88,10000.00,1.0
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12 05:19:53,0,US,0.00,10000.00,0.0
7,The Wild World of Ogots - YOU Become an Ogot (...,Publishing,Publishing,USD,2017-12-20,2017-11-20 23:06:11,45,US,1305.00,1075.00,1.0
8,"VIGATAR, the fitness-machine that lets you pla...",Hardware,Technology,EUR,2014-08-28,2014-06-29 23:30:46,22,NL,4395.53,131879.15,0.0
9,Escape Room Palm Springs - Real Life Escape Game,Live Games,Games,USD,2015-09-21,2015-08-21 17:34:28,88,US,2553.00,7500.00,0.0
10,The Grrrl has a new album called 'Through Ruin...,Music,Music,USD,2010-08-01,2010-06-15 19:00:14,4,US,185.00,178.00,1.0
11,The Tile House,Mixed Media,Art,USD,2012-06-18,2012-04-20 00:53:38,113,US,7360.00,7000.00,1.0


In [9]:
### Создадим вещественную колонку таргет2
data = data.rename({'Собрано в долларах': 'таргет2'}, axis=1)
data.head(10)

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,таргет2,Цель в долларах,таргет1
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,23,US,600.00,600.00,1.0
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,20,US,502.00,10000.00,0.0
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,62,US,2414.00,1400.00,1.0
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03 16:17:21,86,US,10030.88,10000.00,1.0
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12 05:19:53,0,US,0.00,10000.00,0.0
7,The Wild World of Ogots - YOU Become an Ogot (...,Publishing,Publishing,USD,2017-12-20,2017-11-20 23:06:11,45,US,1305.00,1075.00,1.0
8,"VIGATAR, the fitness-machine that lets you pla...",Hardware,Technology,EUR,2014-08-28,2014-06-29 23:30:46,22,NL,4395.53,131879.15,0.0
9,Escape Room Palm Springs - Real Life Escape Game,Live Games,Games,USD,2015-09-21,2015-08-21 17:34:28,88,US,2553.00,7500.00,0.0
10,The Grrrl has a new album called 'Through Ruin...,Music,Music,USD,2010-08-01,2010-06-15 19:00:14,4,US,185.00,178.00,1.0
11,The Tile House,Mixed Media,Art,USD,2012-06-18,2012-04-20 00:53:38,113,US,7360.00,7000.00,1.0


### Чтобы получить матрицу объектов, зачастую нужно обработать сырые данные, то есть извлечь из имеющихся таблиц признаки там, где они не даны явно

In [10]:
### Оставим от колонок с датами разницу в днях 
data['Срок'] = (pd.to_datetime(data['Дедлайн']) - pd.to_datetime(data['Дата публикации'])).dt.days
data.head(5)

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,таргет2,Цель в долларах,таргет1,Срок
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,23,US,600.00,600.0,1.0,31
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,20,US,502.00,10000.0,0.0,29
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,62,US,2414.00,1400.0,1.0,29
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03 16:17:21,86,US,10030.88,10000.0,1.0,44
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12 05:19:53,0,US,0.00,10000.0,0.0,29


In [11]:
### Выделим год запуска проекта
data['Год запуска'] = pd.to_datetime(data['Дата публикации']).dt.year
data.head(5)


,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,таргет2,Цель в долларах,таргет1,Срок,Год запуска
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,23,US,600.00,600.0,1.0,31,2012
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,20,US,502.00,10000.0,0.0,29,2016
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,62,US,2414.00,1400.0,1.0,29,2015
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03 16:17:21,86,US,10030.88,10000.0,1.0,44,2013
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12 05:19:53,0,US,0.00,10000.0,0.0,29,2012


### Иногда может казаться, что признаков мало. Стоит подумать над тем, что еще может объяснять таргет-переменную.

In [12]:
macro = pd.read_excel('macrofeatures.xlsx', engine='openpyxl')
macro.head(5)

,Unnamed: 0,Close_brent,Close_sugar,Close_cereals,Close_index_moex,Close_index_moex_10,Close_index_RGBI,Close_index_RTS_oil_and_gas,Close_index_RTS_metallurgy,Close_index_RTS_consumer_sector,Close_index_RTS_telecom,Close_index_RTS_finance,Close_index_RTS_transport,Close_index_RTS_chemicals,Close_index_RTS_broad_market,Close_index_RTS_electricity,dlk_cob_date
0,0,34.41,13.97,442.75,1797.27,3940.81,125.59,123.40,111.97,196.55,70.17,140.57,27.06,177.38,530.59,32.49,2016-02-24
1,1,35.06,14.24,445.25,1803.89,3977.35,126.44,124.22,112.51,198.03,70.56,142.64,27.43,179.48,536.20,33.07,2016-02-25
2,2,35.13,14.00,443.25,1816.73,4027.23,126.90,125.38,113.44,200.13,71.94,145.45,28.06,181.56,544.73,33.55,2016-02-26
3,3,36.64,14.36,445.00,1840.17,4084.24,126.87,126.69,114.66,200.32,72.41,147.22,28.49,186.76,552.82,34.41,2016-02-29
4,4,36.60,14.39,438.50,1844.17,4087.06,127.78,129.72,117.09,204.30,74.26,150.04,30.12,190.67,565.45,34.96,2016-03-01


In [13]:
macro.columns

Index(['Unnamed: 0', 'Close_brent', 'Close_sugar', 'Close_cereals',
       'Close_index_moex', 'Close_index_moex_10', 'Close_index_RGBI',
       'Close_index_RTS_oil_and_gas', 'Close_index_RTS_metallurgy',
       'Close_index_RTS_consumer_sector', 'Close_index_RTS_telecom',
       'Close_index_RTS_finance', 'Close_index_RTS_transport',
       'Close_index_RTS_chemicals', 'Close_index_RTS_broad_market',
       'Close_index_RTS_electricity', 'dlk_cob_date'],
      dtype='str')

In [14]:
### Смерджим колонки. Оставим от макрофичей какую-нибудь одну.
macro = macro[['Close_brent', 'dlk_cob_date']].drop_duplicates()

In [15]:
macro['dlk_cob_date'] = pd.to_datetime(macro['dlk_cob_date'])

In [16]:
data['Дата публикации'] = pd.to_datetime(data['Дата публикации']).dt.date.astype('datetime64[ns]')
data.head(5)

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,таргет2,Цель в долларах,таргет1,Срок,Год запуска
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09,23,US,600.00,600.0,1.0,31,2012
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19,20,US,502.00,10000.0,0.0,29,2016
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17,62,US,2414.00,1400.0,1.0,29,2015
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03,86,US,10030.88,10000.0,1.0,44,2013
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12,0,US,0.00,10000.0,0.0,29,2012


In [17]:
data = pd.merge(data, macro, how="left", left_on='Дата публикации', right_on='dlk_cob_date').drop(columns='dlk_cob_date')
data.head(5)

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,таргет2,Цель в долларах,таргет1,Срок,Год запуска,Close_brent
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09,23,US,600.00,600.0,1.0,31,2012,NaN
1,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19,20,US,502.00,10000.0,0.0,29,2016,52.54
2,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17,62,US,2414.00,1400.0,1.0,29,2015,NaN
3,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03,86,US,10030.88,10000.0,1.0,44,2013,NaN
4,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12,0,US,0.00,10000.0,0.0,29,2012,NaN


In [18]:
data['Close_brent'].value_counts()

Close_brent
49.25    542
51.61    452
56.45    418
49.04    396
49.74    368
        ... 
64.56      3
66.87      1
65.99      1
66.16      1
66.46      1
Name: count, Length: 404, dtype: int64

In [19]:
data = data.sort_values(['Дата публикации'])
data.head(5)

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,таргет2,Цель в долларах,таргет1,Срок,Год запуска,Close_brent
176128,Grace Jones Does Not Give A F$#% T-Shirt (limi...,Fashion,Fashion,USD,2009-05-31,2009-04-21,30,US,625.0,1000.0,0.0,39,2009,NaN
241929,CRYSTAL ANTLERS UNTITLED MOVIE,Shorts,Film & Video,USD,2009-07-20,2009-04-23,3,US,22.0,80000.0,0.0,87,2009,NaN
244460,drawing for dollars,Illustration,Art,USD,2009-05-03,2009-04-24,3,US,35.0,20.0,1.0,8,2009,NaN
80845,Offline Wikipedia iPhone app,Software,Technology,USD,2009-07-14,2009-04-25,25,US,145.0,99.0,1.0,79,2009,NaN
181197,Pantshirts,Fashion,Fashion,USD,2009-05-26,2009-04-27,10,US,387.0,1900.0,0.0,28,2009,NaN


In [20]:
data['Close_brent'] = data['Close_brent'].ffill()

In [21]:
data['Close_brent'].dropna()

149221    34.41
72755     34.41
235246    34.41
182888    34.41
72631     34.41
          ...  
247730    65.25
318187    66.46
264474    65.99
293634    66.16
23315     66.87
Name: Close_brent, Length: 84850, dtype: float64

In [22]:
data['Close_brent'] = data['Close_brent'].fillna(34.31)
data.head()

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,таргет2,Цель в долларах,таргет1,Срок,Год запуска,Close_brent
176128,Grace Jones Does Not Give A F$#% T-Shirt (limi...,Fashion,Fashion,USD,2009-05-31,2009-04-21,30,US,625.0,1000.0,0.0,39,2009,34.31
241929,CRYSTAL ANTLERS UNTITLED MOVIE,Shorts,Film & Video,USD,2009-07-20,2009-04-23,3,US,22.0,80000.0,0.0,87,2009,34.31
244460,drawing for dollars,Illustration,Art,USD,2009-05-03,2009-04-24,3,US,35.0,20.0,1.0,8,2009,34.31
80845,Offline Wikipedia iPhone app,Software,Technology,USD,2009-07-14,2009-04-25,25,US,145.0,99.0,1.0,79,2009,34.31
181197,Pantshirts,Fashion,Fashion,USD,2009-05-26,2009-04-27,10,US,387.0,1900.0,0.0,28,2009,34.31


In [23]:
### Уберем колонки Дедлайн, Дата запуска, Инвесторы, Название, Страна
data = data.drop(['Дедлайн', 'Дата публикации', 'Инвесторов', 'Название', 'Страна'], axis=1)
data.head(5)


,Категория,Главная категория,Валюта,таргет2,Цель в долларах,таргет1,Срок,Год запуска,Close_brent
176128,Fashion,Fashion,USD,625.0,1000.0,0.0,39,2009,34.31
241929,Shorts,Film & Video,USD,22.0,80000.0,0.0,87,2009,34.31
244460,Illustration,Art,USD,35.0,20.0,1.0,8,2009,34.31
80845,Software,Technology,USD,145.0,99.0,1.0,79,2009,34.31
181197,Fashion,Fashion,USD,387.0,1900.0,0.0,28,2009,34.31


## One-hot encoding

In [24]:
### One-hot encoding колонки с валютой
vals = pd.get_dummies(data['Валюта'], dtype=int)
data = pd.concat([data, vals], axis=1).drop(columns='Валюта')
data.head(5)


,Категория,Главная категория,таргет2,Цель в долларах,таргет1,Срок,Год запуска,Close_brent,AUD,CAD,...,EUR,GBP,HKD,JPY,MXN,NOK,NZD,SEK,SGD,USD
176128,Fashion,Fashion,625.0,1000.0,0.0,39,2009,34.31,0,0,...,0,0,0,0,0,0,0,0,0,1
241929,Shorts,Film & Video,22.0,80000.0,0.0,87,2009,34.31,0,0,...,0,0,0,0,0,0,0,0,0,1
244460,Illustration,Art,35.0,20.0,1.0,8,2009,34.31,0,0,...,0,0,0,0,0,0,0,0,0,1
80845,Software,Technology,145.0,99.0,1.0,79,2009,34.31,0,0,...,0,0,0,0,0,0,0,0,0,1
181197,Fashion,Fashion,387.0,1900.0,0.0,28,2009,34.31,0,0,...,0,0,0,0,0,0,0,0,0,1


In [25]:
data = data.drop(columns='AUD')

In [26]:
### Аналогично закодируем Главную категорию
main_categ = pd.get_dummies(data['Главная категория'], dtype=int)
data = pd.concat([data, main_categ], axis=1).drop(['Games', 'Главная категория'], axis=1)
data.head(5)

,Категория,таргет2,Цель в долларах,таргет1,Срок,Год запуска,Close_brent,CAD,CHF,DKK,...,Design,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater
176128,Fashion,625.0,1000.0,0.0,39,2009,34.31,0,0,0,...,0,1,0,0,0,0,0,0,0,0
241929,Shorts,22.0,80000.0,0.0,87,2009,34.31,0,0,0,...,0,0,1,0,0,0,0,0,0,0
244460,Illustration,35.0,20.0,1.0,8,2009,34.31,0,0,0,...,0,0,0,0,0,0,0,0,0,0
80845,Software,145.0,99.0,1.0,79,2009,34.31,0,0,0,...,0,0,0,0,0,0,0,0,1,0
181197,Fashion,387.0,1900.0,0.0,28,2009,34.31,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [27]:
data.head()

,Категория,таргет2,Цель в долларах,таргет1,Срок,Год запуска,Close_brent,CAD,CHF,DKK,...,Design,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater
176128,Fashion,625.0,1000.0,0.0,39,2009,34.31,0,0,0,...,0,1,0,0,0,0,0,0,0,0
241929,Shorts,22.0,80000.0,0.0,87,2009,34.31,0,0,0,...,0,0,1,0,0,0,0,0,0,0
244460,Illustration,35.0,20.0,1.0,8,2009,34.31,0,0,0,...,0,0,0,0,0,0,0,0,0,0
80845,Software,145.0,99.0,1.0,79,2009,34.31,0,0,0,...,0,0,0,0,0,0,0,0,1,0
181197,Fashion,387.0,1900.0,0.0,28,2009,34.31,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [28]:
### А что с колонкой Категория?
categs = data.groupby(['Категория'])['таргет2'].mean()
data['Категория'] = data['Категория'].map(categs)
data['Категория']

176128     6035.989239
241929     3591.033473
244460     3661.424550
80845      4321.245721
181197     6035.989239
              ...     
247730     7635.064778
318187    38415.722876
264474     6098.303122
293634    38415.722876
23315      1313.020012
Name: Категория, Length: 331675, dtype: float64

## Определимся с таргетом

In [29]:
data = data.drop('таргет1', axis=1)

In [37]:
data.head()

,Категория,таргет2,Цель в долларах,Срок,Год запуска,Close_brent,CAD,CHF,DKK,EUR,...,Design,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater
176128,6035.989239,625.0,1000.0,39,2009,34.31,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
241929,3591.033473,22.0,80000.0,87,2009,34.31,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
244460,3661.424550,35.0,20.0,8,2009,34.31,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
80845,4321.245721,145.0,99.0,79,2009,34.31,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
181197,6035.989239,387.0,1900.0,28,2009,34.31,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [30]:
### Разделим данные на объекты и ответы
X = data.drop('таргет2', axis = 1)
Y = data['таргет2']


In [31]:
Y

176128    625.00
241929     22.00
244460     35.00
80845     145.00
181197    387.00
           ...  
247730    269.84
318187    544.77
264474    202.00
293634    265.00
23315     513.60
Name: таргет2, Length: 331675, dtype: float64

In [32]:
X

,Категория,Цель в долларах,Срок,Год запуска,Close_brent,CAD,CHF,DKK,EUR,GBP,...,Design,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater
176128,6035.989239,1000.00,39,2009,34.31,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
241929,3591.033473,80000.00,87,2009,34.31,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
244460,3661.424550,20.00,8,2009,34.31,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
80845,4321.245721,99.00,79,2009,34.31,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
181197,6035.989239,1900.00,28,2009,34.31,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247730,7635.064778,35.98,2,2017,65.25,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
318187,38415.722876,271.03,4,2017,66.46,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
264474,6098.303122,200.00,3,2017,65.99,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
293634,38415.722876,250.00,1,2017,66.16,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [33]:
## Сейчас будет магия!
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, Y)

X['Предсказание'] = model.predict(X)

X.head()

,Категория,Цель в долларах,Срок,Год запуска,Close_brent,CAD,CHF,DKK,EUR,GBP,...,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater,Предсказание
176128,6035.989239,1000.0,39,2009,34.31,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,4630.592614
241929,3591.033473,80000.0,87,2009,34.31,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,5966.321246
244460,3661.424550,20.0,8,2009,34.31,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,-136.549931
80845,4321.245721,99.0,79,2009,34.31,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,6048.225850
181197,6035.989239,1900.0,28,2009,34.31,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,3775.170109


In [34]:
data.head(5)


,Категория,таргет2,Цель в долларах,Срок,Год запуска,Close_brent,CAD,CHF,DKK,EUR,...,Design,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater
176128,6035.989239,625.0,1000.0,39,2009,34.31,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
241929,3591.033473,22.0,80000.0,87,2009,34.31,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
244460,3661.424550,35.0,20.0,8,2009,34.31,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
80845,4321.245721,145.0,99.0,79,2009,34.31,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
181197,6035.989239,387.0,1900.0,28,2009,34.31,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [35]:
###MSE
(((X['Предсказание'] - Y)**2).mean())**(1/2)


np.float64(95954.43325361362)

In [36]:
###MAE
(abs(X['Предсказание'] - Y)).mean()

np.float64(13651.101864937216)